# BugBuster — ESP32-S3 Flash Manager

**Architecture:** PlatformIO (`esp32s3` env) · Vite/Preact web UI · BBP v4 · `bugbuster` Python library (USB + HTTP)  
**Last refreshed:** 2026-05-04

Builds and flashes the ESP32-S3 firmware and web UI bundle, then connects via
the Python library to verify the device.

| Section | Action |
|---|---|
| 1 | Build web UI (Vite/pnpm) |
| 2 | Build firmware (PlatformIO) |
| 3 | Flash firmware + SPIFFS (needs board) |
| 4 | Connect via Python library + read device info |
| 5 | HTTP transport (optional) |
| 6 | Feature demo: USB-PD + fault check |

> **Hardware steps** (sections 3+) are marked. They require a connected ESP32-S3 board.
> Sections 1–2 are hardware-free.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_ROOT    = Path(os.getcwd()).parent
FW_DIR       = REPO_ROOT / 'Firmware' / 'ESP32'
WEB_DIR      = FW_DIR / 'web'
PIO_ENV      = 'esp32s3'

print(f'Repo root  : {REPO_ROOT}')
print(f'Firmware   : {FW_DIR}')
print(f'Web UI     : {WEB_DIR}')
print(f'FW exists  : {FW_DIR.exists()}')

## 1. Build web UI

The on-device browser UI lives in `Firmware/ESP32/web/` (Vite + Preact + TypeScript).
`pnpm build` compiles and minifies it into `Firmware/ESP32/data/`, which
`pio run -t uploadfs` then packages into the `spiffs` partition.

Requires **pnpm** (`npm i -g pnpm`) and **Node >= 18**.

In [ ]:
!cd "{WEB_DIR}" && pnpm install && pnpm build

## 2. Build firmware (PlatformIO)

Compiles the ESP32-S3 firmware. No board connection required.

In [ ]:
!cd "{FW_DIR}" && pio run -e {PIO_ENV}

## 3. Flash firmware + SPIFFS

> **User must run** — requires a connected ESP32-S3 board.

The two cells below upload firmware and the SPIFFS web bundle.
They are commented out to prevent accidental execution on kernel restart.
Uncomment and run when your board is connected.

In [ ]:
# Run in a terminal (or uncomment here when board is connected):
# !cd "{FW_DIR}" && pio run -e {PIO_ENV} -t upload
print('Flash firmware: uncomment the line above and run when board is connected.')

In [ ]:
# Run in a terminal (or uncomment here when board is connected):
# !cd "{FW_DIR}" && pio run -e {PIO_ENV} -t uploadfs
print('Flash SPIFFS (web UI): uncomment the line above and run when board is connected.')

## 4. Connect via Python library

Uses `bugbuster.connect_usb()` over the BBP v4 binary protocol.
The cell is wrapped in `try/except` so the notebook loads cleanly without a board.

In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT / 'python'))

import bugbuster
print('bugbuster version:', bugbuster.__version__)

In [ ]:
# Auto-detect USB port or override with BUGBUSTER_PORT env var
import os

USB_PORT = os.environ.get('BUGBUSTER_PORT', None)

if USB_PORT is None:
    # Attempt auto-detection via pyserial
    try:
        import serial.tools.list_ports
        ESPRESSIF_VID = 0x303A
        ports = [p.device for p in serial.tools.list_ports.comports()
                 if p.vid == ESPRESSIF_VID]
        USB_PORT = ports[0] if ports else None
        if USB_PORT:
            print(f'Auto-detected port: {USB_PORT}')
        else:
            print('No Espressif USB port found. Set BUGBUSTER_PORT env var.')
    except ImportError:
        print('pyserial not installed. Set BUGBUSTER_PORT env var.')
else:
    print(f'Using port from env: {USB_PORT}')

print(f'USB_PORT = {USB_PORT!r}')

In [ ]:
bb = None

try:
    if USB_PORT is None:
        raise RuntimeError('No USB port configured — skipping device connection.')

    bb = bugbuster.connect_usb(USB_PORT)

    # Device info
    fw_major, fw_minor, fw_patch = bb.get_firmware_version()
    mac = bb.get_mac_address()
    info = bb.get_device_info()
    ping = bb.ping()

    print(f'Firmware version : {fw_major}.{fw_minor}.{fw_patch}')
    print(f'MAC address      : {mac}')
    print(f'Uptime           : {ping.uptime_ms} ms')
    print(f'Device info      : spi_ok={info.spi_ok}  silicon_rev={info.silicon_rev}')

except Exception as e:
    print(f'[No board] {e}')
    print('Connect a BugBuster board and set USB_PORT to run this cell.')

## 5. HTTP transport (optional)

If `BUGBUSTER_HTTP_HOST` is set, connect over WiFi instead.
Use `BUGBUSTER_ADMIN_TOKEN` to authenticate destructive endpoints.

In [ ]:
import os

HTTP_HOST  = os.environ.get('BUGBUSTER_HTTP_HOST', None)
ADMIN_TOKEN = os.environ.get('BUGBUSTER_ADMIN_TOKEN', None)

bb_http = None

try:
    if HTTP_HOST is None:
        raise RuntimeError('BUGBUSTER_HTTP_HOST not set — skipping HTTP transport.')

    bb_http = bugbuster.connect_http(HTTP_HOST)
    if ADMIN_TOKEN:
        bb_http._admin_token = ADMIN_TOKEN

    fw = bb_http.get_firmware_version()
    mac = bb_http.get_mac_address()
    print(f'HTTP connected to {HTTP_HOST}')
    print(f'Firmware: {fw[0]}.{fw[1]}.{fw[2]}   MAC: {mac}')

except Exception as e:
    print(f'[HTTP transport skipped] {e}')

## 6. Feature demo: device status, USB-PD, fault check

Demonstrates three common operations using the USB transport (`bb`).

In [ ]:
try:
    if bb is None:
        raise RuntimeError('No device connected — run section 4 first.')

    # Device status
    status = bb.get_status()
    print('--- Device status ---')
    for k, v in status.items():
        print(f'  {k}: {v}')

    # USB-PD
    usbpd = bb.usbpd_get_status()
    print('\n--- USB-PD status ---')
    for k, v in usbpd.items():
        print(f'  {k}: {v}')

    # Fault check
    faults = bb.check_faults()
    print('\n--- Active faults ---')
    if faults:
        for f in faults:
            print(f'  {f}')
    else:
        print('  No active faults.')

except Exception as e:
    print(f'[No board] {e}')

In [ ]:
# Clean up connections
for dev in [bb, bb_http]:
    if dev is not None:
        try:
            dev.disconnect()
        except Exception:
            pass
print('Connections closed.')